## **FASE 01**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
def cargar_datos(ruta_excel: Path, hoja: str = "Data") -> pd.DataFrame:
    df = pd.read_excel(ruta_excel, sheet_name=hoja)
    return df


def limpiar_y_transformar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Estandarización de columnas esperadas
    columnas_esperadas = [
        "Ceco", "Fecha", "Hora", "Bateria",
        "Velocidad", "RPM_Promedio", "RPM_Min", "RPM_Max"
    ]
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas esperadas: {faltantes}")

    # Tipos de dato
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df["Hora"] = pd.to_numeric(df["Hora"], errors="coerce")
    for c in ["Consumo_Total", "Bateria", "Velocidad", "RPM_Promedio", "RPM_Min", "RPM_Max"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Remover filas sin fecha/hora/ceco
    df = df.dropna(subset=["Fecha", "Hora", "Ceco"]).copy()

    # Acotar hora a 0-23 cuando corresponda
    df = df[(df["Hora"] >= 0) & (df["Hora"] <= 23)].copy()

    # Reglas de negocio de actividad
    df["Equipo_Activo"] = (
        (df["RPM_Promedio"].fillna(0) > 0) |
        (df["Velocidad"].fillna(0) > 0)
    ).astype(int)

    # Consumo operativo
    df["Consumo_Operativo"] = np.where(
        df["Equipo_Activo"] == 1,
        df["Consumo_Total"],
        np.nan
    )

    # Variables temporales
    df["DiaSemana"] = df["Fecha"].dt.dayofweek
    df["Mes"] = df["Fecha"].dt.month
    df["DiaMes"] = df["Fecha"].dt.day
    df["EsFinSemana"] = df["DiaSemana"].isin([5, 6]).astype(int)

    # Variables auxiliares
    df["RPM_Rango"] = df["RPM_Max"] - df["RPM_Min"]
    df["Bateria_Baja"] = (df["Bateria"] < 20).astype(int)

    # Outliers físicos imposibles
    df.loc[df["Bateria"] < 0, "Bateria"] = np.nan
    df.loc[df["Bateria"] > 100, "Bateria"] = np.nan
    for c in ["Velocidad", "RPM_Promedio", "RPM_Min", "RPM_Max"]:
        df.loc[df[c] < 0, c] = np.nan

    # Orden temporal
    df = df.sort_values(["Ceco", "Fecha", "Hora"]).reset_index(drop=True)
    return df


def generar_resumen_calidad(df: pd.DataFrame) -> pd.DataFrame:
    resumen = pd.DataFrame({
        "variable": df.columns,
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean().values * 100).round(2),
    })
    return resumen.sort_values("pct_nulos", ascending=False)

In [ ]:
ruta_excel = Path("/content/drive/MyDrive/MAESTRIA/TERCER SEMESTRE/UNIR_EQUIPO08B/Seminario/Datasets/Dataset-Sensores.xlsx")   # cambia aquí la ruta si está en otra carpeta
hoja = "Data"
outdir = Path("salida")
outdir.mkdir(parents=True, exist_ok=True)

df = cargar_datos(ruta_excel, hoja)
limpio = limpiar_y_transformar(df)
resumen = generar_resumen_calidad(limpio)

limpio.to_csv(outdir / "fase1_dataset_limpio.csv", index=False, encoding="utf-8-sig")
resumen.to_csv(outdir / "fase1_resumen_calidad.csv", index=False, encoding="utf-8-sig")

print("Proceso completado.")
print(f"Filas procesadas: {len(limpio):,}")
print(f"Equipos (CECO) únicos: {limpio['Ceco'].nunique():,}")
print(f"Dataset limpio guardado en: {outdir / 'fase1_dataset_limpio.csv'}")
print(f"Resumen de calidad guardado en: {outdir / 'fase1_resumen_calidad.csv'}")

Proceso completado.
Filas procesadas: 67,475
Equipos (CECO) únicos: 21
Dataset limpio guardado en: salida/fase1_dataset_limpio.csv
Resumen de calidad guardado en: salida/fase1_resumen_calidad.csv


In [ ]:
display(limpio.head())
display(resumen.head(10))

,Ceco,Fecha,Hora,Consumo_Total,Bateria,Velocidad,RPM_Promedio,RPM_Min,RPM_Max,Equipo_Activo,Consumo_Operativo,DiaSemana,Mes,DiaMes,EsFinSemana,RPM_Rango,Bateria_Baja
0,29037,2025-01-06,23,55.21,0.0,0.0,1434.0,1434.0,1438.0,1,55.21,0,1,6,0,4,1
1,29037,2025-01-07,0,25.63,0.0,0.0,1428.0,1423.0,1434.0,1,25.63,1,1,7,0,11,1
2,29037,2025-01-07,1,-28.11,0.0,0.0,1334.0,1233.0,1430.0,1,-28.11,1,1,7,0,197,1
3,29037,2025-01-07,2,-1.85,0.0,0.0,1353.0,1233.0,1396.0,1,-1.85,1,1,7,0,163,1
4,29037,2025-01-07,3,-1.70,0.0,0.0,1391.0,1382.0,1396.0,1,-1.70,1,1,7,0,14,1


,variable,nulos,pct_nulos
10,Consumo_Operativo,31555,46.77
4,Bateria,26456,39.21
5,Velocidad,234,0.35
0,Ceco,0,0.00
2,Hora,0,0.00
3,Consumo_Total,0,0.00
6,RPM_Promedio,0,0.00
7,RPM_Min,0,0.00
1,Fecha,0,0.00
8,RPM_Max,0,0.00


## **FASE 02**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
def preparar_datos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df = df[(df["Equipo_Activo"] == 1) & (df["Consumo_Operativo"].notna())].copy()
    df = df.sort_values(["Fecha", "Hora"]).reset_index(drop=True)
    return df


def split_temporal(df: pd.DataFrame, proporcion_train: float = 0.8):
    corte = int(len(df) * proporcion_train)
    train = df.iloc[:corte].copy()
    test = df.iloc[corte:].copy()
    return train, test


def construir_modelo(nombre_modelo, num_features, cat_features):
    if nombre_modelo == "ridge":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        model = Ridge(alpha=1.0)

    elif nombre_modelo == "random_forest":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])

        model = RandomForestRegressor(
            n_estimators=250,
            max_depth=12,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        )

    else:
        raise ValueError(f"Modelo no soportado: {nombre_modelo}")

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    pre = ColumnTransformer([
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features),
    ])

    pipe = Pipeline([
        ("preprocess", pre),
        ("model", model),
    ])
    return pipe


def metricas_regresion(y_true, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(
        np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))
    ) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "MAPE_pct": mape
    }


def comparar_modelos(X_train, y_train, X_test, y_test, features_num, features_cat):
    modelos = {
        "Ridge": construir_modelo("ridge", features_num, features_cat),
        "RandomForest": construir_modelo("random_forest", features_num, features_cat),
    }

    resultados = []
    predicciones = {}
    pipes_entrenados = {}

    for nombre, pipe in modelos.items():
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        met = metricas_regresion(y_test, y_pred)
        met["Modelo"] = nombre

        resultados.append(met)
        predicciones[nombre] = y_pred
        pipes_entrenados[nombre] = pipe

    metricas_df = pd.DataFrame(resultados)
    metricas_df = metricas_df[["Modelo", "RMSE", "MAE", "R2", "MAPE_pct"]]
    metricas_df = metricas_df.sort_values(by=["MAE", "RMSE"], ascending=[True, True]).reset_index(drop=True)

    mejor_modelo = metricas_df.loc[0, "Modelo"]

    return metricas_df, predicciones, pipes_entrenados, mejor_modelo

In [ ]:
ruta_input = Path("salida/fase1_dataset_limpio.csv")
outdir = Path("salida")
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(ruta_input)
df = preparar_datos(df)

features_num = [
    "Hora", "Bateria", "Velocidad", "RPM_Promedio",
    "RPM_Min", "RPM_Max", "RPM_Rango",
    "DiaSemana", "Mes", "DiaMes", "EsFinSemana"
]
features_cat = ["Ceco"]
target = "Consumo_Operativo"

train, test = split_temporal(df, proporcion_train=0.8)

X_train = train[features_num + features_cat]
y_train = train[target]
X_test = test[features_num + features_cat]
y_test = test[target]

print("Train:", train.shape)
print("Test:", test.shape)

Train: (28736, 17)
Test: (7184, 17)


In [ ]:
metricas_df, predicciones, pipes_entrenados, mejor_modelo = comparar_modelos(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    features_num=features_num,
    features_cat=features_cat
)

test = test.copy()
test["Consumo_Esperado_Ridge"] = predicciones["Ridge"]
test["Consumo_Esperado_RandomForest"] = predicciones["RandomForest"]

col_mejor = f"Consumo_Esperado_{mejor_modelo}"
test["Modelo_Seleccionado"] = mejor_modelo
test["Consumo_Esperado"] = test[col_mejor]

print("Mejor modelo seleccionado:", mejor_modelo)
display(metricas_df)

Mejor modelo seleccionado: RandomForest


,Modelo,RMSE,MAE,R2,MAPE_pct
0,RandomForest,17.823074,4.730671,0.229281,2.041854e+07
1,Ridge,20.111947,4.846584,0.018615,2.159105e+07


In [ ]:
salida_pred = outdir / "fase2_predicciones.csv"
test.to_csv(salida_pred, index=False, encoding="utf-8-sig")

salida_metricas_csv = outdir / "fase2_comparacion_modelos.csv"
metricas_df.to_csv(salida_metricas_csv, index=False, encoding="utf-8-sig")

salida_metricas_txt = outdir / "fase2_metricas_modelo.txt"
with open(salida_metricas_txt, "w", encoding="utf-8") as f:
    f.write("Comparación de modelos de regresión\n")
    f.write("==================================\n\n")
    for _, row in metricas_df.iterrows():
        f.write(f"Modelo: {row['Modelo']}\n")
        f.write(f"RMSE: {row['RMSE']:.6f}\n")
        f.write(f"MAE: {row['MAE']:.6f}\n")
        f.write(f"R2: {row['R2']:.6f}\n")
        f.write(f"MAPE_pct: {row['MAPE_pct']:.6f}\n")
        f.write("-" * 40 + "\n")

    f.write(f"\nModelo seleccionado: {mejor_modelo}\n")

print("Modelos entrenados y evaluados.")
print(f"Predicciones guardadas en: {salida_pred}")
print(f"Comparación de modelos guardada en: {salida_metricas_csv}")
print(f"Resumen de métricas guardado en: {salida_metricas_txt}")

Modelos entrenados y evaluados.
Predicciones guardadas en: salida/fase2_predicciones.csv
Comparación de modelos guardada en: salida/fase2_comparacion_modelos.csv
Resumen de métricas guardado en: salida/fase2_metricas_modelo.txt


In [ ]:
display(test.head())

display(
    test[
        [
            "Fecha", "Hora", "Ceco", "Consumo_Operativo",
            "Consumo_Esperado_Ridge",
            "Consumo_Esperado_RandomForest",
            "Modelo_Seleccionado",
            "Consumo_Esperado"
        ]
    ].head(20)
)

,Ceco,Fecha,Hora,Consumo_Total,Bateria,Velocidad,RPM_Promedio,RPM_Min,RPM_Max,Equipo_Activo,...,DiaSemana,Mes,DiaMes,EsFinSemana,RPM_Rango,Bateria_Baja,Consumo_Esperado_Ridge,Consumo_Esperado_RandomForest,Modelo_Seleccionado,Consumo_Esperado
28736,61526,2025-04-11,11,0.00,0.00,0.0,1435.0,1150.0,1451.0,1,...,4,4,11,0,301,1,-0.454501,0.075018,RandomForest,0.075018
28737,61535,2025-04-11,11,-2.95,0.04,0.0,1437.0,1432.0,1454.0,1,...,4,4,11,0,22,1,-2.567200,-0.149153,RandomForest,-0.149153
28738,29037,2025-04-11,12,-0.04,0.00,0.0,1305.0,0.0,1446.0,1,...,4,4,11,0,1446,1,2.941878,1.019270,RandomForest,1.019270
28739,29063,2025-04-11,12,-2.99,0.04,0.0,1725.0,1714.0,1750.0,1,...,4,4,11,0,36,1,-1.801622,-0.609327,RandomForest,-0.609327
28740,29113,2025-04-11,12,-1.75,1.02,0.0,961.0,0.0,1471.0,1,...,4,4,11,0,1471,1,1.586292,0.993533,RandomForest,0.993533


,Fecha,Hora,Ceco,Consumo_Operativo,Consumo_Esperado_Ridge,Consumo_Esperado_RandomForest,Modelo_Seleccionado,Consumo_Esperado
28736,2025-04-11,11,61526,0.00,-0.454501,0.075018,RandomForest,0.075018
28737,2025-04-11,11,61535,-2.95,-2.567200,-0.149153,RandomForest,-0.149153
28738,2025-04-11,12,29037,-0.04,2.941878,1.019270,RandomForest,1.019270
28739,2025-04-11,12,29063,-2.99,-1.801622,-0.609327,RandomForest,-0.609327
28740,2025-04-11,12,29113,-1.75,1.586292,0.993533,RandomForest,0.993533
28741,2025-04-11,12,29346,0.00,-0.976436,-0.315094,RandomForest,-0.315094
28742,2025-04-11,12,29347,-4.06,-0.997142,-1.126651,RandomForest,-1.126651
28743,2025-04-11,12,29348,0.00,-0.837671,-1.002920,RandomForest,-1.002920
28744,2025-04-11,12,61526,0.00,1.913713,0.983485,RandomForest,0.983485
28745,2025-04-11,12,61535,-2.34,-0.629208,1.122393,RandomForest,1.122393


# **FASE 03**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
def mad(series: pd.Series) -> float:
    med = np.median(series)
    return float(np.median(np.abs(series - med)))


def calcular_umbrales(df: pd.DataFrame) -> pd.DataFrame:
    filas = []

    for ceco, g in df.groupby("Ceco"):
        abs_res = g["Residuo_Abs"].dropna()
        if len(abs_res) == 0:
            continue

        med = float(np.median(abs_res))
        mad_val = mad(abs_res)
        umbral = med + 3 * mad_val

        filas.append({
            "Ceco": ceco,
            "Mediana_Residuo_Abs": med,
            "MAD_Residuo_Abs": mad_val,
            "Umbral_Anomalia": umbral
        })

    return pd.DataFrame(filas)

In [ ]:
ruta_input = Path("salida/fase2_predicciones.csv")
outdir = Path("salida")
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(ruta_input)

df["Residuo"] = df["Consumo_Operativo"] - df["Consumo_Esperado"]
df["Residuo_Abs"] = np.abs(df["Residuo"])
df["Residuo_Pct"] = (
    df["Residuo_Abs"] / np.clip(np.abs(df["Consumo_Esperado"]), 1e-6, None)
) * 100

df.head()

,Ceco,Fecha,Hora,Consumo_Total,Bateria,Velocidad,RPM_Promedio,RPM_Min,RPM_Max,Equipo_Activo,...,EsFinSemana,RPM_Rango,Bateria_Baja,Consumo_Esperado_Ridge,Consumo_Esperado_RandomForest,Modelo_Seleccionado,Consumo_Esperado,Residuo,Residuo_Abs,Residuo_Pct
0,61526,2025-04-11,11,0.00,0.00,0.0,1435.0,1150.0,1451.0,1,...,0,301,1,-0.454501,0.075018,RandomForest,0.075018,-0.075018,0.075018,100.000000
1,61535,2025-04-11,11,-2.95,0.04,0.0,1437.0,1432.0,1454.0,1,...,0,22,1,-2.567200,-0.149153,RandomForest,-0.149153,-2.800847,2.800847,1877.839756
2,29037,2025-04-11,12,-0.04,0.00,0.0,1305.0,0.0,1446.0,1,...,0,1446,1,2.941878,1.019270,RandomForest,1.019270,-1.059270,1.059270,103.924376
3,29063,2025-04-11,12,-2.99,0.04,0.0,1725.0,1714.0,1750.0,1,...,0,36,1,-1.801622,-0.609327,RandomForest,-0.609327,-2.380673,2.380673,390.705049
4,29113,2025-04-11,12,-1.75,1.02,0.0,961.0,0.0,1471.0,1,...,0,1471,1,1.586292,0.993533,RandomForest,0.993533,-2.743533,2.743533,276.139160


In [ ]:
umbrales = calcular_umbrales(df)
umbrales.head()

,Ceco,Mediana_Residuo_Abs,MAD_Residuo_Abs,Umbral_Anomalia
0,29037,1.324278,0.667791,3.327650
1,29063,1.890324,0.981818,4.835779
2,29064,1.297266,0.835512,3.803802
3,29072,1.668561,1.259236,5.446270
4,29086,1.118135,1.026272,4.196950


In [ ]:
df = df.merge(umbrales[["Ceco", "Umbral_Anomalia"]], on="Ceco", how="left")

# Fallback global en caso de umbral faltante
umbral_global = float(
    np.median(df["Residuo_Abs"].dropna()) + 3 * mad(df["Residuo_Abs"].dropna())
)

df["Umbral_Anomalia"] = df["Umbral_Anomalia"].fillna(umbral_global)

df["Es_Anomalia"] = (df["Residuo_Abs"] > df["Umbral_Anomalia"]).astype(int)
df["Direccion_Desvio"] = np.where(df["Residuo"] > 0, "Sobreconsumo", "Subconsumo")

df.head()

,Ceco,Fecha,Hora,Consumo_Total,Bateria,Velocidad,RPM_Promedio,RPM_Min,RPM_Max,Equipo_Activo,...,Consumo_Esperado_Ridge,Consumo_Esperado_RandomForest,Modelo_Seleccionado,Consumo_Esperado,Residuo,Residuo_Abs,Residuo_Pct,Umbral_Anomalia,Es_Anomalia,Direccion_Desvio
0,61526,2025-04-11,11,0.00,0.00,0.0,1435.0,1150.0,1451.0,1,...,-0.454501,0.075018,RandomForest,0.075018,-0.075018,0.075018,100.000000,0.933433,0,Subconsumo
1,61535,2025-04-11,11,-2.95,0.04,0.0,1437.0,1432.0,1454.0,1,...,-2.567200,-0.149153,RandomForest,-0.149153,-2.800847,2.800847,1877.839756,4.933065,0,Subconsumo
2,29037,2025-04-11,12,-0.04,0.00,0.0,1305.0,0.0,1446.0,1,...,2.941878,1.019270,RandomForest,1.019270,-1.059270,1.059270,103.924376,3.327650,0,Subconsumo
3,29063,2025-04-11,12,-2.99,0.04,0.0,1725.0,1714.0,1750.0,1,...,-1.801622,-0.609327,RandomForest,-0.609327,-2.380673,2.380673,390.705049,4.835779,0,Subconsumo
4,29113,2025-04-11,12,-1.75,1.02,0.0,961.0,0.0,1471.0,1,...,1.586292,0.993533,RandomForest,0.993533,-2.743533,2.743533,276.139160,4.239848,0,Subconsumo


In [ ]:
salida_anom = outdir / "fase3_anomalias.csv"
salida_umb = outdir / "fase3_umbrales_por_ceco.csv"

df.to_csv(salida_anom, index=False, encoding="utf-8-sig")
umbrales.to_csv(salida_umb, index=False, encoding="utf-8-sig")

tasa = df["Es_Anomalia"].mean() * 100

print("Detección de anomalías completada.")
print(f"Anomalías detectadas: {df['Es_Anomalia'].sum():,}")
print(f"Tasa de anomalía: {tasa:.2f}%")
print(f"Archivo de anomalías: {salida_anom}")
print(f"Archivo de umbrales: {salida_umb}")

Detección de anomalías completada.
Anomalías detectadas: 798
Tasa de anomalía: 11.11%
Archivo de anomalías: salida/fase3_anomalias.csv
Archivo de umbrales: salida/fase3_umbrales_por_ceco.csv


In [ ]:
display(df[[
    "Fecha", "Hora", "Ceco",
    "Consumo_Operativo", "Consumo_Esperado",
    "Residuo", "Residuo_Abs", "Residuo_Pct",
    "Umbral_Anomalia", "Es_Anomalia", "Direccion_Desvio"
]].head(20))

display(umbrales.head(20))

,Fecha,Hora,Ceco,Consumo_Operativo,Consumo_Esperado,Residuo,Residuo_Abs,Residuo_Pct,Umbral_Anomalia,Es_Anomalia,Direccion_Desvio
0,2025-04-11,11,61526,0.00,0.075018,-0.075018,0.075018,100.000000,0.933433,0,Subconsumo
1,2025-04-11,11,61535,-2.95,-0.149153,-2.800847,2.800847,1877.839756,4.933065,0,Subconsumo
2,2025-04-11,12,29037,-0.04,1.019270,-1.059270,1.059270,103.924376,3.327650,0,Subconsumo
3,2025-04-11,12,29063,-2.99,-0.609327,-2.380673,2.380673,390.705049,4.835779,0,Subconsumo
4,2025-04-11,12,29113,-1.75,0.993533,-2.743533,2.743533,276.139160,4.239848,0,Subconsumo
5,2025-04-11,12,29346,0.00,-0.315094,0.315094,0.315094,100.000000,3.969096,0,Sobreconsumo
6,2025-04-11,12,29347,-4.06,-1.126651,-2.933349,2.933349,260.359950,4.022596,0,Subconsumo
7,2025-04-11,12,29348,0.00,-1.002920,1.002920,1.002920,100.000000,1.452367,0,Sobreconsumo
8,2025-04-11,12,61526,0.00,0.983485,-0.983485,0.983485,100.000000,0.933433,1,Subconsumo
9,2025-04-11,12,61535,-2.34,1.122393,-3.462393,3.462393,308.483116,4.933065,0,Subconsumo


,Ceco,Mediana_Residuo_Abs,MAD_Residuo_Abs,Umbral_Anomalia
0,29037,1.324278,0.667791,3.327650
1,29063,1.890324,0.981818,4.835779
2,29064,1.297266,0.835512,3.803802
3,29072,1.668561,1.259236,5.446270
4,29086,1.118135,1.026272,4.196950
5,29113,1.428015,0.937278,4.239848
6,29115,1.978128,1.119179,5.335666
7,29120,1.339522,0.949982,4.189467
8,29340,1.482656,0.953782,4.344002
9,29346,1.810078,0.719672,3.969096


### **Anomalías por CECO**

In [ ]:
df.groupby("Ceco")["Es_Anomalia"].agg(["count", "sum", "mean"]).rename(
    columns={"count": "Registros", "sum": "Anomalias", "mean": "Tasa"}
).sort_values("Tasa", ascending=False)

,Registros,Anomalias,Tasa
Ceco,,,
29115,195,59,0.302564
29352,179,45,0.251397
61534,221,45,0.203620
61526,365,71,0.194521
61539,321,61,0.190031
29037,120,19,0.158333
29086,765,119,0.155556
61540,253,34,0.134387
29347,569,64,0.112478


# **FASE 04**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
def clasificar_reglas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Tipo_Alerta_Regla"] = "Normal"

    mask_anom = df["Es_Anomalia"] == 1

    crit = mask_anom & (df["Bateria"] < 20) & (df["Residuo_Abs"] > df["Umbral_Anomalia"] * 1.2)
    ralenti = mask_anom & (df["Velocidad"] == 0) & (df["RPM_Promedio"] > 0) & (df["Direccion_Desvio"] == "Sobreconsumo")
    exigente = mask_anom & (df["Velocidad"] > df["Velocidad"].median()) & (df["Direccion_Desvio"] == "Sobreconsumo")
    subcons = mask_anom & (df["Direccion_Desvio"] == "Subconsumo")

    df.loc[crit, "Tipo_Alerta_Regla"] = "Critica_Bateria_y_Consumo"
    df.loc[ralenti, "Tipo_Alerta_Regla"] = "Consumo_Ineficiente_en_Ralenti"
    df.loc[exigente, "Tipo_Alerta_Regla"] = "Operacion_Exigente_Sobreconsumo"
    df.loc[subcons, "Tipo_Alerta_Regla"] = "Subconsumo_Atipico"
    df.loc[mask_anom & (df["Tipo_Alerta_Regla"] == "Normal"), "Tipo_Alerta_Regla"] = "Anomalia_No_Clasificada"

    prioridad = {
        "Critica_Bateria_y_Consumo": "Alta",
        "Consumo_Ineficiente_en_Ralenti": "Alta",
        "Operacion_Exigente_Sobreconsumo": "Media",
        "Subconsumo_Atipico": "Media",
        "Anomalia_No_Clasificada": "Media",
        "Normal": "Baja",
    }

    df["Prioridad"] = df["Tipo_Alerta_Regla"].map(prioridad)
    return df


def clustering_auxiliar(df: pd.DataFrame, n_clusters: int = 3) -> pd.DataFrame:
    df = df.copy()
    anom = df[df["Es_Anomalia"] == 1].copy()

    if len(anom) < n_clusters:
        df["Cluster_Anomalia"] = -1
        return df

    features = ["Residuo_Abs", "Residuo_Pct", "Velocidad", "RPM_Promedio", "Bateria"]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    X = pipe.fit_transform(anom[features])

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    anom["Cluster_Anomalia"] = km.fit_predict(X)

    df["Cluster_Anomalia"] = -1
    df.loc[anom.index, "Cluster_Anomalia"] = anom["Cluster_Anomalia"]

    return df

In [ ]:
ruta_input = Path("salida/fase3_anomalias.csv")
outdir = Path("salida")
outdir.mkdir(parents=True, exist_ok=True)

n_clusters = 3

In [ ]:
df = pd.read_csv(ruta_input)
df.head()

,Ceco,Fecha,Hora,Consumo_Total,Bateria,Velocidad,RPM_Promedio,RPM_Min,RPM_Max,Equipo_Activo,...,Consumo_Esperado_Ridge,Consumo_Esperado_RandomForest,Modelo_Seleccionado,Consumo_Esperado,Residuo,Residuo_Abs,Residuo_Pct,Umbral_Anomalia,Es_Anomalia,Direccion_Desvio
0,61526,2025-04-11,11,0.00,0.00,0.0,1435.0,1150.0,1451.0,1,...,-0.454501,0.075018,RandomForest,0.075018,-0.075018,0.075018,100.000000,0.933433,0,Subconsumo
1,61535,2025-04-11,11,-2.95,0.04,0.0,1437.0,1432.0,1454.0,1,...,-2.567200,-0.149153,RandomForest,-0.149153,-2.800847,2.800847,1877.839756,4.933065,0,Subconsumo
2,29037,2025-04-11,12,-0.04,0.00,0.0,1305.0,0.0,1446.0,1,...,2.941878,1.019270,RandomForest,1.019270,-1.059270,1.059270,103.924376,3.327650,0,Subconsumo
3,29063,2025-04-11,12,-2.99,0.04,0.0,1725.0,1714.0,1750.0,1,...,-1.801622,-0.609327,RandomForest,-0.609327,-2.380673,2.380673,390.705049,4.835779,0,Subconsumo
4,29113,2025-04-11,12,-1.75,1.02,0.0,961.0,0.0,1471.0,1,...,1.586292,0.993533,RandomForest,0.993533,-2.743533,2.743533,276.139160,4.239848,0,Subconsumo


In [ ]:
df = clasificar_reglas(df)
df[["Es_Anomalia", "Tipo_Alerta_Regla", "Prioridad"]].head(20)

,Es_Anomalia,Tipo_Alerta_Regla,Prioridad
0,0,Normal,Baja
1,0,Normal,Baja
2,0,Normal,Baja
3,0,Normal,Baja
4,0,Normal,Baja
5,0,Normal,Baja
6,0,Normal,Baja
7,0,Normal,Baja
8,1,Subconsumo_Atipico,Media
9,0,Normal,Baja


In [ ]:
df = clustering_auxiliar(df, n_clusters=n_clusters)
df[["Es_Anomalia", "Tipo_Alerta_Regla", "Cluster_Anomalia"]].head(20)

,Es_Anomalia,Tipo_Alerta_Regla,Cluster_Anomalia
0,0,Normal,-1
1,0,Normal,-1
2,0,Normal,-1
3,0,Normal,-1
4,0,Normal,-1
5,0,Normal,-1
6,0,Normal,-1
7,0,Normal,-1
8,1,Subconsumo_Atipico,0
9,0,Normal,-1


In [ ]:
resumen = (
    df.groupby(["Tipo_Alerta_Regla", "Prioridad"], dropna=False)
      .size()
      .reset_index(name="Cantidad")
      .sort_values("Cantidad", ascending=False)
)

resumen

,Tipo_Alerta_Regla,Prioridad,Cantidad
3,Normal,Baja,6386
5,Subconsumo_Atipico,Media,611
1,Consumo_Ineficiente_en_Ralenti,Alta,176
4,Operacion_Exigente_Sobreconsumo,Media,8
2,Critica_Bateria_y_Consumo,Alta,2
0,Anomalia_No_Clasificada,Media,1


In [ ]:
salida_alertas = outdir / "fase4_alertas_clasificadas.csv"
salida_resumen = outdir / "fase4_resumen_alertas.csv"

df.to_csv(salida_alertas, index=False, encoding="utf-8-sig")
resumen.to_csv(salida_resumen, index=False, encoding="utf-8-sig")

print("Clasificación operativa completada.")
print(f"Alertas guardadas en: {salida_alertas}")
print(f"Resumen guardado en: {salida_resumen}")

Clasificación operativa completada.
Alertas guardadas en: salida/fase4_alertas_clasificadas.csv
Resumen guardado en: salida/fase4_resumen_alertas.csv


In [ ]:
display(df.head(20))
display(resumen.head(20))

,Ceco,Fecha,Hora,Consumo_Total,Bateria,Velocidad,RPM_Promedio,RPM_Min,RPM_Max,Equipo_Activo,...,Consumo_Esperado,Residuo,Residuo_Abs,Residuo_Pct,Umbral_Anomalia,Es_Anomalia,Direccion_Desvio,Tipo_Alerta_Regla,Prioridad,Cluster_Anomalia
0,61526,2025-04-11,11,0.00,0.00,0.0,1435.0,1150.0,1451.0,1,...,0.075018,-0.075018,0.075018,100.000000,0.933433,0,Subconsumo,Normal,Baja,-1
1,61535,2025-04-11,11,-2.95,0.04,0.0,1437.0,1432.0,1454.0,1,...,-0.149153,-2.800847,2.800847,1877.839756,4.933065,0,Subconsumo,Normal,Baja,-1
2,29037,2025-04-11,12,-0.04,0.00,0.0,1305.0,0.0,1446.0,1,...,1.019270,-1.059270,1.059270,103.924376,3.327650,0,Subconsumo,Normal,Baja,-1
3,29063,2025-04-11,12,-2.99,0.04,0.0,1725.0,1714.0,1750.0,1,...,-0.609327,-2.380673,2.380673,390.705049,4.835779,0,Subconsumo,Normal,Baja,-1
4,29113,2025-04-11,12,-1.75,1.02,0.0,961.0,0.0,1471.0,1,...,0.993533,-2.743533,2.743533,276.139160,4.239848,0,Subconsumo,Normal,Baja,-1
5,29346,2025-04-11,12,0.00,NaN,0.0,1479.0,1380.0,1525.0,1,...,-0.315094,0.315094,0.315094,100.000000,3.969096,0,Sobreconsumo,Normal,Baja,-1
6,29347,2025-04-11,12,-4.06,NaN,0.0,1686.0,1682.0,1692.0,1,...,-1.126651,-2.933349,2.933349,260.359950,4.022596,0,Subconsumo,Normal,Baja,-1
7,29348,2025-04-11,12,0.00,0.04,0.0,1550.0,1546.0,1553.0,1,...,-1.002920,1.002920,1.002920,100.000000,1.452367,0,Sobreconsumo,Normal,Baja,-1
8,61526,2025-04-11,12,0.00,0.00,0.0,1247.0,0.0,1439.0,1,...,0.983485,-0.983485,0.983485,100.000000,0.933433,1,Subconsumo,Subconsumo_Atipico,Media,0
9,61535,2025-04-11,12,-2.34,0.01,0.0,490.0,0.0,1457.0,1,...,1.122393,-3.462393,3.462393,308.483116,4.933065,0,Subconsumo,Normal,Baja,-1


,Tipo_Alerta_Regla,Prioridad,Cantidad
3,Normal,Baja,6386
5,Subconsumo_Atipico,Media,611
1,Consumo_Ineficiente_en_Ralenti,Alta,176
4,Operacion_Exigente_Sobreconsumo,Media,8
2,Critica_Bateria_y_Consumo,Alta,2
0,Anomalia_No_Clasificada,Media,1
